# Ninai × LangChain Adapter

Demonstrates `ninai.adapters.langchain` — the official LangChain integration
shipped **inside the Ninai SDK** (`pip install "ninai[langchain]"`):

- `NinaiChatMessageHistory` — persists conversation turns in Ninai memory
- `NinaiSearchTool` / `NinaiMemoryTool` — LangChain `BaseTool` wrappers
- `get_ninai_toolkit` — convenience factory for agent executors

All API calls are **mocked** — runs offline without a live Ninai server.

In [1]:
import sys, os
SDK_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))
if SDK_PATH not in sys.path:
    sys.path.insert(0, SDK_PATH)

import langchain_core
import ninai
from ninai.adapters.langchain import (
    NinaiChatMessageHistory,
    NinaiSearchTool,
    NinaiMemoryTool,
    get_ninai_toolkit,
)

print('langchain_core :', langchain_core.__version__)
print('ninai SDK      :', ninai.__version__)
print('Adapter imports OK.')


langchain_core : 1.2.28
ninai SDK      : 0.0.1b1
Adapter imports OK.


## Mock Ninai client

In production replace `client` with `NinaiClient(api_key=os.environ['NINAI_API_KEY'])`.

In [2]:
from unittest.mock import MagicMock
from types import SimpleNamespace

_STORE: dict = {}
_CTR = [0]

def _mock_create(**kwargs):
    _CTR[0] += 1
    m = SimpleNamespace(
        id=str(_CTR[0]), content=kwargs.get('content', ''),
        title=kwargs.get('title', ''), tags=kwargs.get('tags', []),
    )
    _STORE[m.id] = m
    return m

def _mock_search(query, **kwargs):
    items = [
        SimpleNamespace(memory_id=m.id, content=m.content, score=0.9, title=m.title)
        for m in list(_STORE.values())[-5:]
    ]
    return SimpleNamespace(items=items[:3], total=len(items))

client = MagicMock()
client.memories.create.side_effect = _mock_create
client.memories.search.side_effect = _mock_search
print('Mock client ready')


Mock client ready


## 1. NinaiChatMessageHistory

In [3]:
from langchain_core.messages import HumanMessage, AIMessage

history = NinaiChatMessageHistory(session_id='demo-001', ninai_client=client)
history.add_messages([HumanMessage(content='What is Ninai?')])
history.add_messages([AIMessage(content='Ninai is a Cognitive OS for enterprise.')])
print(f'Messages in history : {len(history.messages)}')
print(f'Ninai writes        : {client.memories.create.call_count}')


Messages in history : 2
Ninai writes        : 2


## 2. Search & Memory tools

In [4]:
search_tool = NinaiSearchTool(ninai_client=client)
remember_tool = NinaiMemoryTool(ninai_client=client)

remember_tool._run('Q3 revenue target is $4.2M', tags='finance,Q3')
remember_tool._run('Deploy freeze starts 2026-04-15', tags='ops,deploy')

print('Search results:')
print(search_tool._run('revenue target'))


Search results:
- [0.90] What is Ninai?
- [0.90] Ninai is a Cognitive OS for enterprise.
- [0.90] Q3 revenue target is $4.2M


## 3. RunnableWithMessageHistory chain

In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage, AIMessage


def stub_llm(prompt_value):
    msgs = prompt_value.to_messages()
    user_text = next((m.content for m in reversed(msgs) if isinstance(m, HumanMessage)), '')
    return AIMessage(content=f'[stub] Received: "{user_text}"')


chain_with_history = RunnableWithMessageHistory(
    ChatPromptTemplate.from_messages([
        ('system', 'You are a helpful enterprise assistant powered by Ninai.'),
        MessagesPlaceholder(variable_name='history'),
        ('human', '{input}'),
    ]) | RunnableLambda(stub_llm),
    lambda sid: NinaiChatMessageHistory(session_id=sid, ninai_client=client),
    input_messages_key='input',
    history_messages_key='history',
)

cfg = {'configurable': {'session_id': 'lc-42'}}
r1 = chain_with_history.invoke({'input': 'What is our Q3 revenue target?'}, config=cfg)
r2 = chain_with_history.invoke({'input': 'When is the next deploy freeze?'}, config=cfg)
print('Turn 1:', r1.content)
print('Turn 2:', r2.content)
print('\nLangChain + Ninai adapter verified.')


Turn 1: [stub] Received: "What is our Q3 revenue target?"
Turn 2: [stub] Received: "When is the next deploy freeze?"

LangChain + Ninai adapter verified.


## 4. Full toolkit

In [6]:
print('Full toolkit:')
for t in get_ninai_toolkit(client):
    print(f'  {t.name:22s} {t.description}')


Full toolkit:
  ninai_search           Search organisational memory for relevant facts, past decisions, or prior conversations. Input: a natural-language query string.
  ninai_remember         Store an important fact, decision, or observation in Ninai organisational memory so it can be recalled later.


## 5. Enterprise feature gating

Community users get a clear `EnterpriseFeatureRequired` exception — not a raw 403 — when they hit an enterprise-only endpoint.

In [7]:
# Enterprise feature gating example.
# If your org does not have an Enterprise license, the server returns 403
# and the SDK raises EnterpriseFeatureRequired with a clear upgrade message.
from ninai.exceptions import EnterpriseFeatureRequired

try:
    # Simulated: calling an enterprise-only endpoint
    raise EnterpriseFeatureRequired(
        'This feature requires an Enterprise license.',
        feature='enterprise.advanced_observability',
    )
except EnterpriseFeatureRequired as e:
    print(f'Caught: {e}')
    print(f'Feature flag: {e.feature}')


Caught: This feature requires an Enterprise license. (feature=enterprise.advanced_observability) — upgrade at https://sansten.com/ninai/pricing
Feature flag: enterprise.advanced_observability
